# TravelAgent Error Counts By Agent

Numbers-only summary from `example_export/manifest.jsonl`, `example_export/tool_calls.jsonl`, and `example_export/langsmith_runs.csv`.

In [1]:
from __future__ import annotations

import json
import math
import re
from pathlib import Path
from urllib.parse import unquote_plus

import numpy as np
import pandas as pd
import plotly.express as px
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option('display.max_columns', 80)
pd.set_option('display.max_colwidth', 140)

ROOT = Path.cwd()
EXPORT = ROOT / 'thread_analysis' / 'travel_agent'
if not EXPORT.exists():
    EXPORT = ROOT / 'example_export'

def read_jsonl(path: Path) -> list[dict]:
    return [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]

def parse_json(value):
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return None
    if isinstance(value, (dict, list)):
        return value
    try:
        return json.loads(str(value))
    except Exception:
        return None

def norm(value) -> str:
    if value is None:
        return ''
    if not isinstance(value, str):
        value = json.dumps(value, ensure_ascii=False, sort_keys=True)
    value = unquote_plus(value.lower())
    value = re.sub(r'https?://\S+', ' ', value)
    value = re.sub(r'[^a-z0-9\s.-]', ' ', value)
    return re.sub(r'\s+', ' ', value).strip()

manifest = pd.DataFrame(read_jsonl(EXPORT / 'manifest.jsonl'))
tool_raw = pd.DataFrame(read_jsonl(EXPORT / 'tool_calls.jsonl'))
runs = pd.read_csv(EXPORT / 'langsmith_runs.csv')

tool_raw['args'] = tool_raw['tool_args_json'].map(parse_json)
tool_raw['args_norm'] = tool_raw['args'].map(norm)
tool_raw['query'] = tool_raw['args'].map(lambda x: (x or {}).get('query') or (x or {}).get('url') or (x or {}).get('title') or json.dumps(x or {}, ensure_ascii=False, sort_keys=True))
tool_raw['query_norm'] = tool_raw['query'].map(norm)

tool_calls = tool_raw.drop_duplicates(['thread_id', 'tool_call_id', 'tool_name', 'args_norm']).copy().reset_index(drop=True)
tool_calls['seq'] = tool_calls.groupby('thread_id').cumcount()

TOOL_TO_AGENT = {
    'init_plan': 'Execution Agent / TravelPlan Tools',
    'add_day': 'Execution Agent / TravelPlan Tools',
    'add_slot': 'Execution Agent / TravelPlan Tools',
    'insert_slot': 'Execution Agent / TravelPlan Tools',
    'delete_slot': 'Execution Agent / TravelPlan Tools',
    'remove_day': 'Execution Agent / TravelPlan Tools',
    'view_plan': 'Execution Agent / TravelPlan Tools',
    'cost_summary': 'Execution Agent / TravelPlan Tools',
    'write_todos': 'Execution Agent / Todo List',
    'search_flights': 'Flight Search Agent',
    'search_hotels': 'Hotel Search Agent',
    'search_restaurants': 'Restaurant Search Agent',
    'search_attractions': 'Attraction Search Agent',
    'search_web': 'General Web Search Agent',
    'build_place_distance_graph': 'Routing Check Agent',
    'check_route_timing': 'Routing Check Agent',
    'distance_between_places': 'Routing Check Agent',
    'closest_places_to_target': 'Routing Check Agent',
}
tool_calls['agent'] = tool_calls['tool_name'].map(TOOL_TO_AGENT).fillna('Other Tool')
tool_raw['agent'] = tool_raw['tool_name'].map(TOOL_TO_AGENT).fillna('Other Tool')

SEARCH_TOOLS = {'search_flights', 'search_hotels', 'search_restaurants', 'search_attractions', 'search_web', 'build_place_distance_graph', 'check_route_timing', 'distance_between_places', 'closest_places_to_target'}
PLAN_TOOLS = {'init_plan', 'add_day', 'add_slot', 'insert_slot', 'delete_slot', 'remove_day', 'view_plan', 'cost_summary', 'write_todos'}
tool_calls['is_search_tool'] = tool_calls['tool_name'].isin(SEARCH_TOOLS)
tool_calls['is_plan_tool'] = tool_calls['tool_name'].isin(PLAN_TOOLS)

print({
    'manifest_traces': len(manifest),
    'flat_runs': len(runs),
    'raw_tool_observations': len(tool_raw),
    'unique_tool_calls': len(tool_calls),
    'unique_tool_call_ids': tool_raw['tool_call_id'].nunique(),
})

{'manifest_traces': 15, 'flat_runs': 2300, 'raw_tool_observations': 2300, 'unique_tool_calls': 2300, 'unique_tool_call_ids': 2300}


## Main Error Table

In [2]:
# 1. repeated tool calls: same agent/tool/normalized-args appears more than once in the same thread
raw_by_agent = tool_raw.groupby('agent').size().rename('raw_tool_observations')
unique_by_agent = tool_calls.groupby('agent').size().rename('unique_tool_calls')
base = pd.concat([raw_by_agent, unique_by_agent], axis=1).fillna(0).astype(int)
base['export_duplicate_observations'] = (base['raw_tool_observations'] - base['unique_tool_calls']).clip(lower=0)
arg_repeat_groups = tool_calls.groupby(['agent', 'thread_id', 'tool_name', 'args_norm']).size().reset_index(name='n')
repeated_tool_calls = arg_repeat_groups[arg_repeat_groups['n'].gt(1)].assign(extra=lambda d: d['n'] - 1).groupby('agent')['extra'].sum().rename('repeated_tool_calls')

# 2. repeated queries: exact duplicate normalized query per agent/tool/thread
q = tool_calls[tool_calls['is_search_tool'] & tool_calls['query_norm'].ne('')]
exact_repeat_groups = q.groupby(['agent', 'thread_id', 'tool_name', 'query_norm']).size().reset_index(name='n')
repeated_queries = exact_repeat_groups[exact_repeat_groups['n'].gt(1)].groupby('agent')['n'].sum().rename('repeated_queries')

# 3. near repeated queries: semantic duplicates among search-like calls with cosine similarity >= .86
near_counts = {}
for (agent, thread_id, tool_name), g in tool_calls[tool_calls['is_search_tool'] & tool_calls['query_norm'].str.len().gt(10)].groupby(['agent', 'thread_id', 'tool_name']):
    if len(g) < 2:
        continue
    tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=1).fit_transform(g['query_norm'])
    sim = cosine_similarity(tfidf)
    count = int(sum(sim[i, j] >= 0.86 for i in range(len(g)) for j in range(i + 1, len(g))))
    near_counts[agent] = near_counts.get(agent, 0) + count
near_repeated_queries = pd.Series(near_counts, name='near_repeated_query_pairs', dtype='int64')

# 4. dead loops: same tool burst length >= 4, plus exact search repeats
loop_rows = []
for thread_id, g in tool_calls.sort_values(['thread_id', 'seq']).groupby('thread_id'):
    g = g.reset_index(drop=True)
    start = 0
    for i in range(1, len(g) + 1):
        boundary = i == len(g) or g.loc[i, 'tool_name'] != g.loc[start, 'tool_name']
        if boundary:
            length = i - start
            if length >= 4:
                loop_rows.append({'agent': g.loc[start, 'agent'], 'dead_loop_candidates': 1, 'dead_loop_tool_calls': length})
            start = i
dead_loop_df = pd.DataFrame(loop_rows)
dead_loop_candidates = dead_loop_df.groupby('agent')['dead_loop_candidates'].sum() if not dead_loop_df.empty else pd.Series(dtype='int64', name='dead_loop_candidates')
dead_loop_tool_calls = dead_loop_df.groupby('agent')['dead_loop_tool_calls'].sum() if not dead_loop_df.empty else pd.Series(dtype='int64', name='dead_loop_tool_calls')

# 5. timeouts and explicit errors from flat LangSmith runs, mapped by run/tool name when possible
ERROR_RE = re.compile(r'timeout|timed out|exception|traceback|failed|failure|error|rate limit|api failed|tool failed|could not|unable to|forbidden|403|empty|missing_info|missing info', re.I)
TIMEOUT_RE = re.compile(r'timeout|timed out|deadline|cancelled|canceled|max.*iterations|max.*steps|recursion', re.I)
run_names = runs['name'].astype(str)
run_agent_default = pd.Series('Execution Agent / Other Runs', index=runs.index)
run_agent_default = run_agent_default.mask(run_names.str.contains('validator|review', case=False, na=False), 'Quality Review Agent')
run_agent_default = run_agent_default.mask(run_names.str.contains('planner', case=False, na=False), 'Task Planning Agent')
run_agent_default = run_agent_default.mask(run_names.str.contains('constraint', case=False, na=False), 'Constraint Extraction Agent')
runs['agent'] = runs['name'].map(TOOL_TO_AGENT).fillna(run_agent_default)
runs['has_error'] = runs['error'].fillna('').astype(str).str.contains(ERROR_RE)
runs['timeout'] = runs['error'].fillna('').astype(str).str.contains(TIMEOUT_RE)
timeouts = runs.groupby('agent')['timeout'].sum().rename('timeouts')
explicit_errors = runs.groupby('agent')['has_error'].sum().rename('explicit_error_runs')

# 6. cascading errors: risky search/dead-loop event followed by later plan mutation in same thread
MISSING_PARAM_RE = re.compile(r'\b(restaurant|hotel|flight|attraction|museum|route|transit|train|ferry|ticket|opening|hours|meal|lunch|dinner|breakfast)\b', re.I)
DATE_RE = re.compile(r'\b(20\d{2}|jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec|january|february|march|april|june|july|august|september|october|november|monday|tuesday|wednesday|thursday|friday|saturday|sunday|day \d+)\b', re.I)
LOCATION_RE = re.compile(r'\b(from|to|in|near|at|rome|italy|airport|station|hotel|city|downtown|center|centre)\b', re.I)
tool_calls['missing_date_hint'] = tool_calls['query'].fillna('').str.contains(MISSING_PARAM_RE) & ~tool_calls['query'].fillna('').str.contains(DATE_RE)
tool_calls['missing_location_hint'] = tool_calls['query'].fillna('').str.contains(MISSING_PARAM_RE) & ~tool_calls['query'].fillna('').str.contains(LOCATION_RE)
risk = tool_calls[tool_calls['is_search_tool'] & (tool_calls['missing_date_hint'] | tool_calls['missing_location_hint'])].copy()
cascade_rows = []
for r in risk.itertuples(index=False):
    later = tool_calls[(tool_calls['thread_id'].eq(r.thread_id)) & (tool_calls['seq'].gt(r.seq)) & (tool_calls['is_plan_tool'])]
    if len(later):
        cascade_rows.append({'agent': r.agent, 'cascading_error_candidates': 1, 'downstream_plan_mutations': len(later)})
cascade_df = pd.DataFrame(cascade_rows)
cascades = (cascade_df.groupby('agent')['cascading_error_candidates'].sum().rename('cascading_error_candidates') if not cascade_df.empty else pd.Series(dtype='int64', name='cascading_error_candidates'))
downstream_mutations = (cascade_df.groupby('agent')['downstream_plan_mutations'].sum().rename('downstream_plan_mutations_after_risk') if not cascade_df.empty else pd.Series(dtype='int64', name='downstream_plan_mutations_after_risk'))

error_table = pd.concat([
    base[['unique_tool_calls', 'raw_tool_observations', 'export_duplicate_observations']],
    repeated_tool_calls,
    repeated_queries,
    near_repeated_queries,
    dead_loop_candidates,
    dead_loop_tool_calls,
    timeouts,
    explicit_errors,
    cascades,
    downstream_mutations,
], axis=1).fillna(0).astype(int).reset_index(names='agent')

error_cols = ['repeated_tool_calls', 'repeated_queries', 'near_repeated_query_pairs', 'dead_loop_candidates', 'dead_loop_tool_calls', 'timeouts', 'explicit_error_runs', 'cascading_error_candidates']
error_table['total_error_signals'] = error_table[error_cols].sum(axis=1)
error_table = error_table.sort_values('total_error_signals', ascending=False)
display(error_table)

/tmp/ipykernel_22142/3487336431.py:59: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  tool_calls['missing_date_hint'] = tool_calls['query'].fillna('').str.contains(MISSING_PARAM_RE) & ~tool_calls['query'].fillna('').str.contains(DATE_RE)
/tmp/ipykernel_22142/3487336431.py:60: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  tool_calls['missing_location_hint'] = tool_calls['query'].fillna('').str.contains(MISSING_PARAM_RE) & ~tool_calls['query'].fillna('').str.contains(LOCATION_RE)


,agent,unique_tool_calls,raw_tool_observations,export_duplicate_observations,repeated_tool_calls,repeated_queries,near_repeated_query_pairs,dead_loop_candidates,dead_loop_tool_calls,timeouts,explicit_error_runs,cascading_error_candidates,downstream_plan_mutations_after_risk,total_error_signals
2,Execution Agent / TravelPlan Tools,1559,1559,0,287,0,0,84,1199,0,0,0,0,1570
6,Restaurant Search Agent,190,190,0,0,0,2,21,118,0,0,160,3235,301
4,General Web Search Agent,214,214,0,0,0,1,21,130,0,0,110,3123,262
7,Routing Check Agent,75,75,0,0,0,5,10,59,0,0,53,713,127
0,Attraction Search Agent,68,68,0,0,0,0,5,33,0,0,2,53,40
3,Flight Search Agent,37,37,0,0,0,0,1,10,0,0,0,0,11
1,Execution Agent / Todo List,123,123,0,5,0,0,0,0,0,0,0,0,5
5,Hotel Search Agent,34,34,0,0,0,0,0,0,0,0,0,0,0


## Error-Type Totals

In [3]:
error_type_totals = error_table[error_cols].sum().rename('count').reset_index().rename(columns={'index': 'error_type'}).sort_values('count', ascending=False)
display(error_type_totals)
fig = px.bar(error_type_totals, x='error_type', y='count', title='Error Signals By Type')
fig.show()

,error_type,count
4,dead_loop_tool_calls,1549
7,cascading_error_candidates,325
0,repeated_tool_calls,292
3,dead_loop_candidates,142
2,near_repeated_query_pairs,8
1,repeated_queries,0
5,timeouts,0
6,explicit_error_runs,0


## Agent Totals

In [4]:
agent_totals = error_table[['agent', 'unique_tool_calls', 'total_error_signals']].sort_values('total_error_signals', ascending=False)
display(agent_totals)
fig = px.bar(error_table, x='agent', y=error_cols, barmode='stack', title='Grouped Error Signals By Agent')
fig.update_layout(xaxis_tickangle=-35)
fig.show()

,agent,unique_tool_calls,total_error_signals
2,Execution Agent / TravelPlan Tools,1559,1570
6,Restaurant Search Agent,190,301
4,General Web Search Agent,214,262
7,Routing Check Agent,75,127
0,Attraction Search Agent,68,40
3,Flight Search Agent,37,11
1,Execution Agent / Todo List,123,5
5,Hotel Search Agent,34,0


## Supporting Tables

In [5]:
tool_detail = tool_calls.groupby(['agent', 'tool_name']).agg(unique_calls=('tool_call_id', 'nunique')).reset_index().sort_values(['agent', 'unique_calls'], ascending=[True, False])
display(tool_detail)

query_repeat_detail = exact_repeat_groups[exact_repeat_groups['n'].gt(1)].sort_values('n', ascending=False)
display(query_repeat_detail[['agent', 'tool_name', 'n', 'query_norm']].head(50))

display(risk[['agent', 'seq', 'tool_name', 'missing_date_hint', 'missing_location_hint', 'query']].head(100))

,agent,tool_name,unique_calls
0,Attraction Search Agent,search_attractions,68
1,Execution Agent / Todo List,write_todos,123
3,Execution Agent / TravelPlan Tools,add_slot,971
2,Execution Agent / TravelPlan Tools,add_day,238
5,Execution Agent / TravelPlan Tools,delete_slot,128
9,Execution Agent / TravelPlan Tools,view_plan,79
7,Execution Agent / TravelPlan Tools,insert_slot,74
4,Execution Agent / TravelPlan Tools,cost_summary,41
6,Execution Agent / TravelPlan Tools,init_plan,26
8,Execution Agent / TravelPlan Tools,remove_day,2


,agent,tool_name,n,query_norm


,agent,seq,tool_name,missing_date_hint,missing_location_hint,query
14,General Web Search Agent,14,search_web,False,True,official Café Museum Vienna opening hours Sunday 08:00 22:00 Karlsplatz
16,General Web Search Agent,16,search_web,False,True,official Naschmarkt Vienna Sunday opening hours market stalls closed
17,General Web Search Agent,17,search_web,False,True,official opening hours Café Sacher Vienna Sunday July 12 2026 daily 08:00 midnight
18,General Web Search Agent,18,search_web,False,True,official opening hours Upper Belvedere Vienna Sunday July 12 2026 daily 09:00 18:00
19,General Web Search Agent,19,search_web,False,True,official Sunday opening hours July 2026 Vollpension Schleifmühlgasse Vienna daily 08:00 20:00
...,...,...,...,...,...,...
755,Routing Check Agent,445,check_route_timing,True,False,"{""destination_address"": ""Shinjuku Station, Tokyo, Japan"", ""origin_address"": ""CITAN Hostel Tokyo, 15-2 Nihombashi Odenmacho, Chuo City, T..."
756,Routing Check Agent,446,check_route_timing,True,False,"{""destination_address"": ""Haneda Airport Terminal 3, Tokyo, Japan"", ""origin_address"": ""CITAN Hostel Tokyo, 15-2 Nihombashi Odenmacho, Chu..."
757,Routing Check Agent,447,check_route_timing,True,False,"{""destination_address"": ""Asakusa Station, Tokyo, Japan"", ""origin_address"": ""CITAN Hostel Tokyo, 15-2 Nihombashi Odenmacho, Chuo City, To..."
758,Routing Check Agent,448,check_route_timing,True,False,"{""destination_address"": ""Akihabara Station, Tokyo, Japan"", ""origin_address"": ""CITAN Hostel Tokyo, 15-2 Nihombashi Odenmacho, Chuo City, ..."
